# Claim-Extraktion mit ClimateBERT

Dieses Notebook extrahiert potentielle Umweltaussagen aus ESG-Berichten mithilfe des in `src/greenwashing_pipeline/claim_extraction.py` implementierten ClimateBERT-Workflows. Die extrahierten Claims werden anschließend in einer CSV-Datei gespeichert, sodass sie im nächsten Schritt weiterverarbeitet werden können.

## Voraussetzungen
- Python-Abhängigkeiten aus `requirements.txt` installieren (inkl. `transformers`, `torch`, `pdfplumber`, `pandas`).
- Die zu analysierenden Nachhaltigkeitsberichte (PDF) liegen im Ordner `ESG Reports` oder an einem frei wählbaren Pfad.

In [ ]:
from pathlib import Path
import pandas as pd

# Stellt sicher, dass der `src`-Ordner importierbar ist
import sys
PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from greenwashing_pipeline.document_loader import PDFDocumentLoader
from greenwashing_pipeline.claim_extraction import ClaimExtractor

## Eingaben konfigurieren
Passe den `pdf_path` und den `output_csv` bei Bedarf an. Mit `max_pages` lässt sich die Seitenzahl zum Testen begrenzen.

In [ ]:
pdf_path = Path("ESG Reports/sustainability-statement-2024.pdf.downloadasset.pdf")
output_csv = Path("data/claims.csv")
max_pages = 5  # auf `None` setzen, um das komplette Dokument zu verarbeiten

output_csv.parent.mkdir(parents=True, exist_ok=True)
print(f"PDF: {pdf_path}")
print(f"Claims werden gespeichert unter: {output_csv}")

## PDF laden und in Abschnitte zerlegen

In [ ]:
loader = PDFDocumentLoader(max_pages=max_pages)
sections = loader.load(pdf_path)
print(f"Geladene Abschnitte: {len(sections)}")
if sections:
    print("Erste Textpassage:
", sections[0].text[:500])

## ClimateBERT-basierte Claim-Extraktion ausführen

In [ ]:
claim_extractor = ClaimExtractor()
claims = claim_extractor.extract_from_sections(sections)
print(f"Gefundene Claims: {len(claims)}")

## Ergebnisse in ein DataFrame überführen und als CSV speichern

In [ ]:
claims_df = pd.DataFrame([
    {
        "document_id": claim.document_id,
        "page": claim.page_number,
        "label": claim.label,
        "score": claim.score,
        "text": claim.text,
    }
    for claim in claims
])

claims_df.to_csv(output_csv, index=False)
claims_df.head()

Die CSV-Datei dient als Eingabe für die LLM-Auswertung und kann bei Bedarf zusätzlich manuell annotiert werden.